#### Transform Customer Data

1. Remove record with NULL customer_id
2. Remove exact Duplicate Records
3. Remove Duplicate Records based on created_timestamp
4. Cast the columns to correct Data Type
5. Write transformed data to the Silver schema

In [0]:
df = spark.sql('select * from gizmobox_catalog_subbu.bronze.py_customers')
display(df)

In [0]:
df=spark.table('gizmobox_catalog_subbu.bronze.py_customers')
display(df)

In [0]:
df=spark.read.table('gizmobox_catalog_subbu.bronze.py_customers')
display(df)

##### 1. Remove record with NULL customer_id

In [0]:
df= spark.table('gizmobox_catalog_subbu.bronze.py_customers')
df_filtered= df.filter('customer_id is not null')
display(df_filtered)

In [0]:
df_filtered = df.filter(df.customer_id.isNotNull())
display(df_filtered)

In [0]:
df_filtered = df.where(df.customer_id.isNotNull())
display(df_filtered)

##### 2. Remove exact Duplicate Records

In [0]:
df_distinct= df_filtered.distinct()
display(df_distinct)


In [0]:
df_distinct = df_filtered.dropDuplicates()
display(df_distinct)

##### 3. Remove Duplicate Records based on created_timestamp

In [0]:
from pyspark.sql import functions as F
df_max_ts =  ( df_distinct.groupBy('customer_id')
              .agg(F.max('created_timestamp').alias('max_created_timestamp'))
             )
display(df_max_ts)

In [0]:
df_distinct_customer = (
                    df_distinct
                    .join(df_max_ts, (df_distinct.customer_id == df_max_ts.customer_id) & 
                                    (df_distinct.created_timestamp == df_max_ts.max_created_timestamp),
                                    'inner')
                    .select(df_distinct['*'])
                    )
display(df_distinct_customer)

##### 4. Cast the columns to correct Data Type

In [0]:
df_casted_customer= (df_distinct_customer
                        .select(df_distinct_customer.created_timestamp.cast('timestamp'),
                                df_distinct_customer.customer_id,
                                df_distinct_customer.customer_name,
                                df_distinct_customer.date_of_birth.cast('date'),
                                df_distinct_customer.email,
                                df_distinct_customer.member_since.cast('date'),
                                df_distinct_customer.telephone

                        )
)
display(df_casted_customer)

##### 5. Write transformed data to the Silver schema (Write data to Delta table)

In [0]:
df_casted_customer.writeTo('gizmobox_catalog_subbu.silver.py_customers').createOrReplace()

In [0]:
display(spark.read.table('gizmobox_catalog_subbu.silver.py_customers'))
